#  Car Type Classifier — Notebook 2 : Prétraitement des données

**Objectif** : Nettoyer les données et construire les descriptions textuelles
qui serviront d'entrée au modèle LLM.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append('../src')
from classifier import load_and_clean_data, build_text_description

print(' Librairies et fonctions chargées')

## 1. Chargement et nettoyage

In [ ]:
df = load_and_clean_data('../data/')
print(f'Dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes')
df.head()

## 2. Nettoyage des valeurs aberrantes

In [ ]:
# Identifier les colonnes numériques pertinentes
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Colonnes numériques :', numeric_cols)

# Supprimer les doublons
n_before = len(df)
df = df.drop_duplicates()
print(f'Doublons supprimés : {n_before - len(df)}')

# Filtrer valeurs aberrantes sur HP si la colonne existe
hp_col = next((c for c in df.columns if 'hp' in c.lower()), None)
if hp_col:
    df[hp_col] = pd.to_numeric(df[hp_col], errors='coerce')
    q_low = df[hp_col].quantile(0.01)
    q_high = df[hp_col].quantile(0.99)
    n_before = len(df)
    df = df[(df[hp_col].isna()) | ((df[hp_col] >= q_low) & (df[hp_col] <= q_high))]
    print(f'Lignes supprimées (HP hors plage [{q_low:.0f}, {q_high:.0f}] HP) : {n_before - len(df)}')

print(f'\nDataset final après nettoyage : {len(df)} lignes')

## 3. Construction des descriptions textuelles

In [ ]:
df['text_description'] = df.apply(build_text_description, axis=1)

print(' Exemples de descriptions générées :')
for i, txt in enumerate(df['text_description'].head(8)):
    print(f'  [{i+1}] {txt}')

In [ ]:
# Statistiques sur les descriptions
df['text_length'] = df['text_description'].str.len()
print('Longueur des descriptions :')
print(df['text_length'].describe())

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
df['text_length'].hist(bins=20, color='#4C72B0', edgecolor='white')
plt.title('Distribution de la longueur des descriptions textuelles', fontweight='bold')
plt.xlabel('Longueur (caractères)')
plt.ylabel('Nombre de voitures')
plt.tight_layout()
plt.show()

## 4. Sauvegarde du dataset prétraité

In [ ]:
os.makedirs('../results', exist_ok=True)
df.to_csv('../results/dataset_preprocessed.csv', index=False)
print(f' Dataset prétraité sauvegardé : results/dataset_preprocessed.csv ({len(df)} lignes)')
df[['text_description']].head()

##  Résumé du prétraitement

| Étape | Action |
|---|---|
| Chargement | Fusion des datasets CSV |
| Nettoyage | Suppression doublons + valeurs aberrantes |
| Normalisation | Colonnes en minuscules, types corrects |
| Construction texte | Concaténation des attributs en une phrase |

 **Passer au notebook 03_classification.ipynb**